# Exploration sim_v1

Notebook d'exploration uniquement : connexion a la base, lecture des tables/vues
via `table_view` / `p_table_view`. Aucune logique metier ici.


## 1. Setup


In [1]:
from pathlib import Path
import sys

# Racine release (dossier de ce notebook)
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline").is_dir():
    # si le kernel a un autre cwd, remonter depuis le fichier
    ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.paths import Paths, release_root
from src.pipeline.connection import PipelineFactory
from src.pipeline.engine import ConnectionPipeline

paths = Paths(ROOT).ensure()
print("ROOT     :", paths.root)
print("DB       :", paths.main_db, "exists=", paths.main_db.exists())
print("pipeline :", paths.pipeline)


ROOT     : /media/laghmari/ssd-data/dev/hotels/release_1_0_0
DB       : /media/laghmari/ssd-data/dev/hotels/release_1_0_0/data/duckdb/main/main.duckdb exists= True
pipeline : /media/laghmari/ssd-data/dev/hotels/release_1_0_0/pipeline


## 2. Connexion pipeline


In [2]:
# Connexion lecture/ecriture sur la base principale + YAML pipeline/
cp = PipelineFactory(paths).open(read_only=False)
print("project_dir :", cp.project_dir)
print("objets pipeline YAML :", len(cp.pipeline))


project_dir : /media/laghmari/ssd-data/dev/hotels/release_1_0_0
objets pipeline YAML : 100


## 3. Relations dans la base


In [3]:
import duckdb

def list_relations(cp, like: str | None = None):
    """Liste tables et vues de la base ouverte."""
    q = """
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = current_schema()
        ORDER BY table_type, table_name
    """
    df = cp.con.execute(q).df()
    if like:
        df = df[df["table_name"].str.contains(like, case=False, na=False)]
    return df

rels = list_relations(cp)
display(rels)
print(f"{len(rels)} relations")


,table_name,table_type
0,__worker_result_buffer,BASE TABLE
1,t_dataset_mix_nombre_detail,BASE TABLE
2,t_dataset_mix_pourcentage_detail,BASE TABLE
3,t_dataset_pivot,BASE TABLE
4,t_hotel_codes,BASE TABLE
...,...,...
64,v_v1_prediction,VIEW
65,v_v1_r1_buyers,VIEW
66,v_v1_r2_mix,VIEW
67,v_v1_r3_categories,VIEW


69 relations


## 4. Filtrer les objets sim_v1

Les tables/vues v1 sont en general prefixees `t_v1_`, `v_v1_`, ou liees aux features hotel.


In [4]:
v1_rels = list_relations(cp, like=r"v1|hotel_param|pilot|scope")
display(v1_rels)


,table_name,table_type
5,t_hotel_params,BASE TABLE
8,t_pilot_concepts,BASE TABLE
9,t_pilot_defaults,BASE TABLE
23,t_scope_hotels,BASE TABLE
24,t_v1_loo_hotels,BASE TABLE
25,t_v1_loo_results,BASE TABLE
39,v_hotel_params,VIEW
60,v_v1_hotel_input,VIEW
61,v_v1_loo_metrics,VIEW
62,v_v1_loo_result,VIEW


## 5. `table_view` — relation deja materialisee


In [5]:
# Exemple : parametres hotel (si la table existe)
name = "t_hotel_params"
if cp.relation_exists(name):
    df = cp.table_view(name).df()
    display(df.head(20))
    print(df.shape, list(df.columns)[:20])
else:
    print(f"{name} absente — lancer le pipeline sim_v1 ou p_table_view ci-dessous")


,hotel_code,solution,label,hotel_name,hotel_brand,nb_chambres,taux_occupation,guests_per_chambre,m_lin,nb_frigos_froid,clients_mois,mix_fb,ca_reel_mensuel,ca_fb_mensuel,ca_nf_mensuel,marge_reelle_mensuelle,nb_ventes_mensuel,nb_paniers_mensuel,n_mois
0,H0373,CONNECTED,MER MONT,Mercure Paris Montmartre Sacré-Cœur,MERCURE,305.0,0.76,2.0,6.0,3.0,14139.80,0.9594,4125.77,3958.40,167.37,21.29,564.43,307.80,28.0
1,H3546,CONNECTED,NOV TE,Novotel Paris Centre Tour Eiffel,NOVOTEL,764.0,0.80,1.8,7.0,3.0,33554.88,0.7414,17839.96,13225.77,4614.20,1396.28,3772.78,1354.70,23.0
2,H6188,LIBERTY,MER BOUL,Mercure Paris Boulogne,MERCURE,191.0,0.73,2.0,6.0,3.0,8505.23,1.0000,1125.05,1125.05,0.00,101.33,353.40,322.83,21.0
3,HB5I0,LIBERTY,NOV MEG,Novotel Megève Mont-Blanc,NOVOTEL,572.0,0.60,1.8,8.0,3.0,18841.68,0.5663,1585.06,897.67,687.39,123.75,223.33,128.40,26.0
4,H2075,SIMPLY,IBB NICE,Ibis budget Nice Californie,IBIS BUDGET,129.0,0.72,1.7,3.0,3.0,4815.83,0.6191,439.71,272.22,167.49,-165.85,138.05,101.05,30.0
5,HB6A3,SIMPLY,IBB STRA,Ibis budget Strasbourg Centre République,IBIS BUDGET,97.0,0.70,1.7,2.0,3.0,3520.61,1.0000,1088.19,1088.19,0.00,-41.52,347.97,164.07,19.0


(6, 19) ['hotel_code', 'solution', 'label', 'hotel_name', 'hotel_brand', 'nb_chambres', 'taux_occupation', 'guests_per_chambre', 'm_lin', 'nb_frigos_froid', 'clients_mois', 'mix_fb', 'ca_reel_mensuel', 'ca_fb_mensuel', 'ca_nf_mensuel', 'marge_reelle_mensuelle', 'nb_ventes_mensuel', 'nb_paniers_mensuel', 'n_mois']


## 6. `p_table_view` — construit avec prerequis automatiques

`process_with_requires` charge les dependances YAML puis renvoie la relation.


In [6]:
# Construit v_hotel_params (et ses requires) si besoin
try:
    df = cp.p_table_view("v_hotel_params").df()
    display(df)
    print(df.shape)
except Exception as exc:
    print("p_table_view v_hotel_params :", exc)


,hotel_code,solution,label,hotel_name,hotel_brand,nb_chambres,taux_occupation,guests_per_chambre,m_lin,nb_frigos_froid,clients_mois,mix_fb,ca_reel_mensuel,ca_fb_mensuel,ca_nf_mensuel,marge_reelle_mensuelle,nb_ventes_mensuel,nb_paniers_mensuel,n_mois
0,H0373,CONNECTED,MER MONT,Mercure Paris Montmartre Sacré-Cœur,MERCURE,305.0,0.76,2.0,6.0,3.0,14139.80,0.9594,4125.77,3958.40,167.37,21.29,564.43,307.80,28.0
1,H3546,CONNECTED,NOV TE,Novotel Paris Centre Tour Eiffel,NOVOTEL,764.0,0.80,1.8,7.0,3.0,33554.88,0.7414,17839.96,13225.77,4614.20,1396.28,3772.78,1354.70,23.0
2,H6188,LIBERTY,MER BOUL,Mercure Paris Boulogne,MERCURE,191.0,0.73,2.0,6.0,3.0,8505.23,1.0000,1125.05,1125.05,0.00,101.33,353.40,322.83,21.0
3,HB5I0,LIBERTY,NOV MEG,Novotel Megève Mont-Blanc,NOVOTEL,572.0,0.60,1.8,8.0,3.0,18841.68,0.5663,1585.06,897.67,687.39,123.75,223.33,128.40,26.0
4,H2075,SIMPLY,IBB NICE,Ibis budget Nice Californie,IBIS BUDGET,129.0,0.72,1.7,3.0,3.0,4815.83,0.6191,439.71,272.22,167.49,-165.85,138.05,101.05,30.0
5,HB6A3,SIMPLY,IBB STRA,Ibis budget Strasbourg Centre République,IBIS BUDGET,97.0,0.70,1.7,2.0,3.0,3520.61,1.0000,1088.19,1088.19,0.00,-41.52,347.97,164.07,19.0


(6, 19)


## 7. Resultats LOO sim_v1


In [7]:
for name in ("t_v1_loo_results", "v_v1_loo_metrics", "t_v1_loo_hotels"):
    print("===", name, "===")
    if not cp.relation_exists(name):
        print("  (absente)")
        continue
    d = cp.table_view(name).df()
    display(d.head(30))
    print("  shape", d.shape)


=== t_v1_loo_results ===


,hotel_code,solution,clients_hotel,taux_acheteurs,nb_acheteurs,mix_steps,mult_fb,mult_nfb,r4_mode,r4_diff,...,ca_nfb_r4,ca_ht_predit,marge_produit_predite,ca_reel_mensuel,marge_reelle_mensuelle,n_mois,erreur_ca,erreur_marge,abs_erreur_ca,abs_erreur_marge
0,H0373,CONNECTED,14139.80,0.112436,1589.824033,2.180,1.48,1.33,frigos_froid,0.0,...,1248.201200,13763.773972,8089.263723,4125.77,21.29,28.0,9638.003972,8067.973723,9638.003972,8067.973723
1,H2075,SIMPLY,4815.83,0.098838,475.986936,-3.809,1.48,1.33,m_lin,1.0,...,157.906285,2291.580028,1362.035395,439.71,-165.85,30.0,1851.870028,1527.885395,1851.870028,1527.885395
2,H3546,CONNECTED,33554.88,0.039918,1339.437681,-2.180,1.48,1.33,frigos_froid,0.0,...,576.779903,13202.170953,7948.472075,17839.96,1396.28,23.0,-4637.789047,6552.192075,4637.789047,6552.192075
3,H6188,LIBERTY,8505.23,0.011853,100.812296,4.337,1.48,1.33,m_lin,-2.0,...,-155.661011,795.829254,537.223881,1125.05,101.33,21.0,-329.220746,435.893881,329.220746,435.893881
4,HB5I0,LIBERTY,18841.68,0.041551,782.888847,-4.337,1.48,1.33,m_lin,2.0,...,815.240347,4156.761462,2309.326311,1585.06,123.75,26.0,2571.701462,2185.576311,2571.701462,2185.576311
5,HB6A3,SIMPLY,3520.61,0.028666,100.921380,3.809,1.48,1.33,m_lin,-1.0,...,22.169893,379.418325,226.725500,1088.19,-41.52,19.0,-708.771675,268.245500,708.771675,268.245500


  shape (6, 21)
=== v_v1_loo_metrics ===


,perimetre,n_hotels,mae_ca,mae_marge,mape_ca_pct,mape_marge_pct
0,ALL,6,3289.559488,3172.961148,156.233484,7021.411070
1,LIBERTY,2,1450.461104,1310.735096,95.754541,1098.147429
2,SIMPLY,2,1280.320852,898.065448,243.145107,783.654341
3,CONNECTED,2,7137.896509,7310.082899,129.800804,19182.431440


  shape (4, 6)
=== t_v1_loo_hotels ===


,hotel_code,solution,label,name
0,H0373,CONNECTED,MER MONT,Mercure Paris Montmartre Sacré-Cœur
1,H3546,CONNECTED,NOV TE,Novotel Paris Centre Tour Eiffel
2,H6188,LIBERTY,MER BOUL,Mercure Paris Boulogne
3,HB5I0,LIBERTY,NOV MEG,Novotel Megève Mont-Blanc
4,H2075,SIMPLY,IBB NICE,Ibis budget Nice Californie
5,HB6A3,SIMPLY,IBB STRA,Ibis budget Strasbourg Centre République


  shape (6, 4)


## 8. Explorer une relation au choix


In [8]:
# Modifier le nom pour inspecter n'importe quelle table/vue
NAME = "v_v1_prediction"  # peut necessiter v_loo_step (iteration)

if cp.relation_exists(NAME):
    try:
        display(cp.table_view(NAME).df().head(50))
    except Exception as exc:
        print("table_view:", exc)
else:
    try:
        display(cp.p_table_view(NAME).df().head(50))
    except Exception as exc:
        print("p_table_view:", exc)


table_view: Catalog Error: Table with name v_loo_step does not exist!
Did you mean "v_loo_result"?


## 9. Fermer la connexion


In [ ]:
cp.close()
print("connexion fermee")
